In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, sum as _sum, count as _count, avg as _avg, desc, date_trunc,
    to_date, current_date, datediff, countDistinct, lit
)

spark = SparkSession.builder.getOrCreate()

fact = spark.table("novocart_global.novocart_gold.fact_sales")

# Considering only completed orders for revenue-related KPIs
completed = fact.filter(col("order_status") == "COMPLETED")


## LOADING TABLES FROM GOLD


In [0]:
from pyspark.sql.functions import col, sum, count, month, year

fact_sales = spark.table("novocart_global.novocart_gold.fact_sales")
dim_product = spark.table("novocart_global.novocart_gold.dim_product")
dim_customer = spark.table("novocart_global.novocart_gold.dim_customer")
dim_country = spark.table("novocart_global.novocart_gold.dim_country")
dim_channel = spark.table("novocart_global.novocart_gold.dim_channel")

In [0]:
kpi_datacube = spark.table("novocart_global.novocart_gold.kpi_datacube")
display(kpi_datacube)

## KPI 1 - Total Revenue

In [0]:
from pyspark.sql.functions import sum

fact_sales = spark.table("novocart_global.novocart_gold.fact_sales")

kpi_total_revenue = fact_sales.select(
    sum("line_total_usd").alias("total_revenue")
)
display(kpi_total_revenue)

## KPI 2 - [Revenue] by Country 

In [0]:
dim_country = spark.table("novocart_global.novocart_gold.dim_country")

kpi_revenue_country = fact_sales \
    .join(dim_country, "country_key") \
    .groupBy("country_code") \
    .agg(sum("line_total_usd").alias("revenue"))
display(kpi_revenue_country)

## KPI 2 - Revenue by Channel


In [0]:
kpi_revenue_channel = fact_sales \
    .join(dim_channel, "channel_key") \
    .groupBy("channel") \
    .agg(sum("line_total_usd").alias("revenue"))
display(kpi_revenue_channel)

## KPI 4 - Completed Order Count 

In [0]:
completed_order_count = fact_sales \
    .filter(col("order_status") == "completed") \
    .select("order_id") \
    .distinct() \
    .count()
display(spark.createDataFrame([(completed_order_count,)], ["completed_order_count"]))

## KPI 5 - Completed Order Rate 

In [0]:
total_orders = fact_sales.select("order_id").distinct().count()
completed_order_rate = completed_order_count / total_orders if total_orders > 0 else None
display(spark.createDataFrame([(completed_order_rate,)], ["completed_order_rate"]))

## KPI 6 - Average Order Value (AOV) 

In [0]:
kpi_AOV = fact_sales \
    .groupBy("order_id") \
    .agg(sum("line_total_usd").alias("order_total")) \
    .select((sum("order_total") / count("*")).alias("avg_order_value"))
display(kpi_AOV)

### KPI 7 - Top 5 Products by Revenue

In [0]:
product_revenue = fact_sales \
    .join(dim_product, "product_key") \
    .groupBy("product_id", "product_name") \
    .agg(sum("line_total_usd").alias("revenue")) \
    .orderBy(col("revenue").desc()) \
    .limit(5)
display(product_revenue)

## KPI 8 -Active Customers Count 

In [0]:
kpi_active_customers = fact_sales \
    .join(dim_customer, "customer_key") \
    .select("customer_id") \
    .distinct() \
    .count()
display(spark.createDataFrame([(kpi_active_customers,)], ["active_customers_count"]))

## KPI 9 - Customer Acquisition by Month 

In [0]:
kpi_customer_acquisition = dim_customer \
    .withColumn("year", year("registration_date")) \
    .withColumn("month", month("registration_date")) \
    .groupBy("year", "month") \
    .count()
display(kpi_customer_acquisition)

## KPI 10 - Data Quality Score 

In [0]:
total_records = fact_sales.count()
valid_records = fact_sales \
    .join(dim_customer, "customer_key") \
    .filter(col("customer_id") != "UNKNOWN") \
    .count()
data_quality_score = valid_records / total_records if total_records > 0 else None
display(spark.createDataFrame([(data_quality_score,)], ["data_quality_score"]))